# Narrator — ElevenLabs TTS Generator

Generates narration audio for each scene using the ElevenLabs API.
A YAML cache prevents redundant API calls: audio is regenerated only when
the narration text, voice, or model changes.

**Voice**: Jack  
**Model**: `eleven_v3` (expressive, audio tags supported)  
**Output**: `audio/scene_NN.mp3`  
**Cache**: `audio/narrator_cache.yaml`

Requires `ELEVENLABS_API_KEY` environment variable.

In [1]:
import hashlib
import os
from datetime import datetime, timezone
from pathlib import Path

import yaml
from elevenlabs.client import ElevenLabs

AUDIO_DIR = Path("audio")
CACHE_FILE = AUDIO_DIR / "narrator_cache.yaml"
MODEL_ID = "eleven_v3"

# ── Voice configuration ────────────────────────────────────────────────────
# Find your voice ID at: https://elevenlabs.io/app/voice-lab
#   1. Open a voice → click "ID" next to the voice name → copy
# Premade voices (always available):
#   Rachel  21m00Tcm4TlvDq8ikWAM  (neutral, clear)
#   Josh    TxGEqnHWrfWFTfGW9XjX  (deep male)
#   Adam    pNInz6obpgDQGcFmaJgB  (narration)
VOICE_NAME = "Jack"
VOICE_ID   = "RL2gbGArFsmr05q4aJLj"   # ← paste your voice ID here
# ──────────────────────────────────────────────────────────────────────────

# ── Scene filter ───────────────────────────────────────────────────────────
# Generate only specific scenes to control credit usage.
# Set to None to generate all scenes.
# Example: SCENES_TO_GENERATE = ["scene_01", "scene_02"]
SCENES_TO_GENERATE: list[str] | None = None
# ──────────────────────────────────────────────────────────────────────────

AUDIO_DIR.mkdir(exist_ok=True)

client = ElevenLabs(api_key=os.environ["ELEVENLABS_API_KEY"])
print(f"Ready. Voice: {VOICE_NAME} ({VOICE_ID}), Model: {MODEL_ID}")

Ready. Voice: Jack (RL2gbGArFsmr05q4aJLj), Model: eleven_v3


In [ ]:
NARRATIONS = {
    "scene_01": {
        "start": "0:00",
        "end": "1:30",
        "text": (
            "[excited] Rust is fast. "
            "Memory-safe, type-checked at compile time, fearless about concurrency. "
            "But the moment you try to build a real web application in Rust, things get rough. "
            "Router, ORM, auth, OpenAPI, migrations — "
            "you end up assembling them yourself, crate by crate. "
            "Meanwhile in Django, one command gives you ORM, admin, auth, and more. "
            "[awe] What if we could keep Rust\'s performance — and get back Django\'s productivity?"
        ),
    },
    "scene_02": {
        "start": "1:30",
        "end": "2:45",
        "text": (
            "[excited] That\'s why we built Reinhardt. "
            "A framework that fuses Django\'s philosophy with Rust\'s type safety. "
            "Its defining trait is a polylithic architecture. "
            "Use the full stack, or pull in just the ORM, or just the DI container — your call. "
            "Today we\'ll build the polls app from the official basics tutorial — "
            "a full-stack WASM application: model, migrations, server functions, a reactive form, and admin. "
            "All the code on screen is in the tutorial. Follow along at your own pace."
        ),
    },
    "scene_03": {
        "start": "2:45",
        "end": "4:00",
        "text": (
            "Two commands scaffold the whole thing. "
            "startproject with the pages template gives you a Rust crate plus a WASM client in one layout. "
            "startapp adds the polls app, wired into config slash apps automatically. "
            "Three pillars to notice. Apps slash holds the Django-style model-view layout. "
            "Shared slash types holds the DTOs used by both sides. "
            "And client slash is your WASM frontend. "
            "[awe] cargo make is the one-stop task runner — migrate, test, build WASM, collect static files, all here."
        ),
    },
    "scene_04": {
        "start": "4:00",
        "end": "5:45",
        "text": (
            "First, the models. "
            "The model attribute turns a plain struct into an ORM entity. "
            "Per-field constraints go in the field attribute — primary key, max length, auto-add timestamps. "
            "[dramatic tone] Every constraint is checked at compile time. "
            "The macro synthesizes typed accessors — Question objects, Question field question text, and friends — "
            "so there are no stringly-typed column names. "
            "Choice adds a foreign key back to Question. "
            "The related name lets you traverse the relation by name, "
            "and the type system knows the shape on both sides. "
            "Migrations: makemigrations emits a Rust file you can git-diff. "
            "migrate applies it. No raw SQL in your review queue."
        ),
    },
    "scene_05": {
        "start": "5:45",
        "end": "8:15",
        "text": (
            "The bridge between WASM and server is the server-fn attribute. "
            "A plain async function, annotated once, becomes a typed RPC the frontend can call directly. "
            "No JSON hand-rolling, no OpenAPI layer. "
            "Arguments marked inject are resolved by the DI container — the database connection just appears. "
            "Return types come from shared slash types. "
            "Backend and frontend share the same Rust type. Schema drift, gone. "
            "On the client, use-action turns the RPC into a reactive resource. "
            "Dispatch it, and a watch block in the page macro re-renders the moment the response arrives. "
            "No JSX, no virtual DOM diff — just Rust expressions inside a macro. "
            "[excited] Register the routes, hit the URL, and the list renders."
        ),
    },
    "scene_06": {
        "start": "8:15",
        "end": "9:45",
        "text": (
            "Forms. The form macro is the single recommended path — there is no separate generic-view layer. "
            "You declare the server function to submit to, "
            "the fields with their widgets and validation, "
            "and the reactive watchers for the submit button, error display, and success navigation. "
            "The macro generates the HTML, wires validation, "
            "and binds every watch block to form state — loading, error, or ok. "
            "[awe] No templating engine. No controller class. Just a declaration that compiles to a WASM component."
        ),
    },
    "scene_07": {
        "start": "9:45",
        "end": "10:45",
        "text": (
            "The admin. Register a model, get a CRUD UI. "
            "Three lines of Rust — one register call per model — "
            "and the admin routes, forms, filters, and pagination all exist. "
            "createsuperuser is the same command, same spirit, as Django. "
            "[awe] Every column, filter, and edit form you see was generated from the model definition alone. "
            "Custom layouts and search live on ModelAdmin — but the default is already production-shaped."
        ),
    },
    "scene_08": {
        "start": "10:45",
        "end": "12:15",
        "text": (
            "The basics tutorial stops here — but polls gets more interesting with authentication. "
            "Reinhardt\'s DI container wires dependencies at compile time. "
            "Each inject argument resolves to a registered factory. "
            "Missing a factory? It\'s a compile error, not a runtime panic. "
            "Guards are injectable too. "
            "A Guard of IsActiveUser checks the user\'s state before your handler body runs. "
            "No token? The auth layer returns 401. "
            "Inactive user? The guard returns 403. "
            "Only active authenticated calls reach your code. "
            "[dramatic tone] Declarative to write, statically enforced. "
            "See the REST tutorial for the full implementation."
        ),
    },
    "scene_09": {
        "start": "12:15",
        "end": "13:30",
        "text": (
            "[awe] Thirteen minutes. That\'s what we just did. "
            "Model, migrations, server functions, reactive UI, forms, admin, DI, auth — "
            "roughly a hundred lines of Rust you wrote yourself. "
            "The same stack assembled by hand out of Axum, sea-orm, utoipa, and a separate frontend framework "
            "would take six hundred lines and a lot of design decisions. "
            "Everything on screen lives in the basics tutorial. Clone it, run it, modify it. "
            "cargo install reinhardt-admin-cli — start today. "
            "[excited] See you next time."
        ),
    },
}

In [3]:
def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()


def load_cache() -> dict:
    if CACHE_FILE.exists():
        with CACHE_FILE.open() as f:
            return yaml.safe_load(f) or {}
    return {}


def save_cache(cache: dict) -> None:
    with CACHE_FILE.open("w") as f:
        yaml.dump(cache, f, default_flow_style=False, allow_unicode=True)


def generate_audio(scene_id: str, scene: dict, cache: dict) -> str:
    """Generate audio for one scene; skip when text/voice/model are unchanged."""
    text = scene["text"]
    cache_key = text_hash(f"{text}|{VOICE_ID}|{MODEL_ID}")
    output_path = AUDIO_DIR / f"{scene_id}.mp3"

    cached = cache.get(scene_id, {})
    if cached.get("cache_key") == cache_key and output_path.exists():
        return "skipped"

    audio_bytes = client.text_to_speech.convert(
        voice_id=VOICE_ID,
        text=text,
        model_id=MODEL_ID,
        output_format="mp3_44100_128",
    )
    output_path.write_bytes(b"".join(audio_bytes))

    cache[scene_id] = {
        "cache_key": cache_key,
        "voice_id": VOICE_ID,
        "voice_name": VOICE_NAME,
        "model_id": MODEL_ID,
        "output_file": str(output_path),
        "start": scene["start"],
        "end": scene["end"],
        "generated_at": datetime.now(timezone.utc).isoformat(),
    }
    return "generated"


# ── Build target list ─────────────────────────────────────────────────────
cache = load_cache()
targets = {
    sid: scene
    for sid, scene in NARRATIONS.items()
    if SCENES_TO_GENERATE is None or sid in SCENES_TO_GENERATE
}

# Credit estimate (1 credit ≈ 1 character for eleven_turbo_v2_5)
# Only counts scenes that would actually be generated (not cached).
to_generate = {
    sid: scene for sid, scene in targets.items()
    if cache.get(sid, {}).get("cache_key") !=
       text_hash(f"{scene['text']}|{VOICE_ID}|{MODEL_ID}")
    or not (AUDIO_DIR / f"{sid}.mp3").exists()
}
est_credits = sum(len(s["text"]) for s in to_generate.values())
print(f"Scenes targeted : {len(targets)}")
print(f"To generate     : {len(to_generate)}  (cached/skipped: {len(targets) - len(to_generate)})")
print(f"Est. credits    : ~{est_credits}")
print()

# ── Run ───────────────────────────────────────────────────────────────────
results = {}
for scene_id, scene in targets.items():
    status = generate_audio(scene_id, scene, cache)
    results[scene_id] = status
    icon = "✓" if status == "skipped" else "🎙"
    print(f"{icon} {scene_id} ({scene['start']}–{scene['end']}): {status}")

save_cache(cache)

# ── Summary ───────────────────────────────────────────────────────────────
n_gen  = sum(1 for v in results.values() if v == "generated")
n_skip = sum(1 for v in results.values() if v == "skipped")
print(f"\nGenerated: {n_gen}  Skipped: {n_skip}")
print()
for mp3 in sorted(AUDIO_DIR.glob("scene_*.mp3")):
    print(f"  {mp3.name}  ({mp3.stat().st_size / 1024:.1f} KB)")

Scenes targeted : 10
To generate     : 10  (cached/skipped: 0)
Est. credits    : ~5853

🎙 scene_01 (0:00–1:30): generated
🎙 scene_02 (1:30–2:30): generated
🎙 scene_03 (2:30–4:00): generated
🎙 scene_04 (4:00–5:30): generated
🎙 scene_05 (5:30–6:30): generated
🎙 scene_06 (6:30–8:00): generated
🎙 scene_07 (8:00–10:30): generated
🎙 scene_08 (10:30–12:00): generated
🎙 scene_09 (12:00–13:30): generated
🎙 scene_10 (13:30–15:00): generated

Generated: 10  Skipped: 0

  scene_01.mp3  (471.1 KB)
  scene_02.mp3  (457.2 KB)
  scene_03.mp3  (826.2 KB)
  scene_04.mp3  (709.8 KB)
  scene_05.mp3  (388.6 KB)
  scene_06.mp3  (929.8 KB)
  scene_07.mp3  (4808.6 KB)
  scene_08.mp3  (411.1 KB)
  scene_09.mp3  (1838.4 KB)
  scene_10.mp3  (553.5 KB)
